# Italian Real Estate Market — Integrated Market Dynamics

This notebook integrates the three analytical dimensions developed in the previous notebooks: **OMI quotations, transaction volumes (NTN) and population**.

The objective is to build a controlled municipality-year/semester analytical panel and investigate whether demographic dynamics are associated with changes in real-estate prices and transaction activity.

**Pipeline:** raw sources → controlled aggregation → key harmonisation → coverage checks → integrated panel → price/volume/demography analysis → interpretation limits.

> Important: association is not causality. The integrated panel is designed for descriptive and exploratory analysis, not causal inference.

## 1. Setup and source inventory

Each source is loaded independently. No notebook state is required from Notebooks 01–03, making this analysis reproducible from the repository data alone.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

QUOTATIONS_DIR = PROJECT_ROOT / 'data' / 'raw' / 'quotations'
TRANSACTIONS_DIR = PROJECT_ROOT / 'data' / 'raw' / 'transactions'
POPULATION_DIR = PROJECT_ROOT / 'data' / 'raw' / 'population'
for path in [QUOTATIONS_DIR, TRANSACTIONS_DIR, POPULATION_DIR]:
    assert path.exists(), f'Missing source directory: {path}'

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Project root:', PROJECT_ROOT)

## 2. OMI quotation panel

OMI observations are aggregated to **municipality + semester**. For residential property, the municipality quotation is represented by the median of the quotation midpoint. This is a descriptive aggregation and is not an official OMI average price.

In [ ]:
quotation_files = sorted(QUOTATIONS_DIR.glob('omi_quotations_*.csv'))
if not quotation_files:
    raise FileNotFoundError('No OMI quotation files found.')

quotation_parts = []
for path in quotation_files:
    match = re.search(r'_(\d{4})_(S[12])$', path.stem)
    if not match:
        continue
    year = int(match.group(1))
    semester = match.group(2)
    df = pd.read_csv(path, sep=';', low_memory=False)
    df.columns = [str(c).replace('\ufeff', '').strip() for c in df.columns]
    required = ['Comune_ISTAT', 'Descr_Tipologia', 'Compr_min', 'Compr_max']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f'{path.name}: missing columns {missing}')
    df['reference_year'] = year
    df['semester'] = semester
    quotation_parts.append(df[required + ['reference_year', 'semester']])

if not quotation_parts:
    raise ValueError('No valid OMI quotation files matched the expected year/semester pattern.')

omi = pd.concat(quotation_parts, ignore_index=True)
omi['municipality_code'] = omi['Comune_ISTAT'].astype('string').str.strip()
for column in ['Compr_min', 'Compr_max']:
    omi[column] = pd.to_numeric(omi[column], errors='coerce')
omi['Compr_mid'] = omi[['Compr_min', 'Compr_max']].mean(axis=1)
residential_mask = omi['Descr_Tipologia'].astype('string').str.contains('abitazion|villa', case=False, na=False)
residential_omi = omi.loc[residential_mask].copy()

quotation_panel = (
    residential_omi.dropna(subset=['municipality_code', 'Compr_mid'])
    .groupby(['reference_year', 'semester', 'municipality_code'], as_index=False)
    .agg(
        median_price_m2=('Compr_mid', 'median'),
        p25_price_m2=('Compr_mid', lambda x: x.quantile(0.25)),
        p75_price_m2=('Compr_mid', lambda x: x.quantile(0.75)),
        quotation_observations=('Compr_mid', 'size'),
    )
)
quotation_panel['reference_date'] = pd.to_datetime(
    quotation_panel['reference_year'].astype(str) + '-' +
    quotation_panel['semester'].map({'S1': '06-30', 'S2': '12-31'})
)
print(f'Integrated quotation observations: {len(quotation_panel):,}')
display(quotation_panel.head())

## 3. Transaction panel

The transaction source is annual. We retain municipalities with a controlled one-row-per-municipality-year structure and identify the NTN field dynamically because source releases can differ in naming.

In [ ]:
year_folders = sorted(p for p in TRANSACTIONS_DIR.iterdir() if p.is_dir() and p.name.isdigit())
transaction_parts = []

def find_file(folder, token):
    matches = [p for p in folder.iterdir() if p.is_file() and token in p.name.lower()]
    if len(matches) != 1:
        raise ValueError(f'{folder.name}: expected one {token} file, found {len(matches)}')
    return matches[0]

for folder in year_folders:
    year = int(folder.name)
    lista = pd.read_csv(find_file(folder, 'lista-com'), sep=';', decimal=',')
    valori_res = pd.read_csv(find_file(folder, 'valori-res'), sep=';', decimal=',')
    lista.columns = [str(c).strip() for c in lista.columns]
    valori_res.columns = [str(c).strip() for c in valori_res.columns]
    cod_lista = next(c for c in lista.columns if re.search(r'codcom$', c, re.I))
    cod_res = next(c for c in valori_res.columns if re.search(r'codcom$', c, re.I))
    lista = lista.rename(columns={cod_lista: 'municipality_code'})
    valori_res = valori_res.rename(columns={cod_res: 'municipality_code'})
    total_candidates = [c for c in valori_res.columns if re.fullmatch(r'NTN_?' + str(year), c, re.I)]
    if not total_candidates:
        total_candidates = [
            c for c in valori_res.columns
            if re.match(r'NTN', c, re.I) and 'mq' not in c.lower()
        ]
    if not total_candidates:
        raise KeyError(f'{year}: NTN total column not found')
    ntn_col = total_candidates[0]
    geography = [c for c in ['municipality_code', 'Comune', 'Provincia', 'Regione'] if c in lista.columns]
    base = lista[geography].copy()
    values = valori_res[['municipality_code', ntn_col]].copy()
    values[ntn_col] = pd.to_numeric(values[ntn_col], errors='coerce')
    if values['municipality_code'].duplicated().any():
        raise ValueError(f'{year}: duplicate municipality transaction keys')
    base = base.merge(values, on='municipality_code', how='left', validate='one_to_one')
    base = base.rename(columns={ntn_col: 'ntn'})
    base['year'] = year
    transaction_parts.append(base)

if not transaction_parts:
    raise ValueError('No transaction year folders found.')

transactions = pd.concat(transaction_parts, ignore_index=True)
assert not transactions.duplicated(['year', 'municipality_code']).any()
print(f'Transaction municipality-years: {len(transactions):,}')
display(transactions.head())

## 4. Population panel

Population is loaded from the official `Età = 999` municipality total. It is annual and therefore joins to both OMI semesters of the same year.

In [ ]:
population_years = sorted(p for p in POPULATION_DIR.iterdir() if p.is_dir() and p.name.isdigit())
population_parts = []
for folder in population_years:
    year = int(folder.name)
    population_files = list(folder.glob('*_Comuni.csv'))
    if len(population_files) != 1:
        raise ValueError(f'{year}: expected one *_Comuni.csv file, found {len(population_files)}')
    path = population_files[0]
    df = pd.read_csv(path, sep=';', encoding='utf-8-sig', usecols=['Codice comune', 'Comune', 'Età', 'Totale'])
    df.columns = ['municipality_code', 'municipality', 'age', 'population']
    df['municipality_code'] = df['municipality_code'].astype('string').str.strip()
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df['population'] = pd.to_numeric(df['population'], errors='coerce')
    totals = df.loc[df['age'].eq(999), ['municipality_code', 'population']].copy()
    totals['year'] = year
    if totals['municipality_code'].duplicated().any():
        raise ValueError(f'{year}: duplicate population municipality keys')
    population_parts.append(totals)

if not population_parts:
    raise ValueError('No population year folders found.')

population = pd.concat(population_parts, ignore_index=True)
print(f'Population municipality-years: {len(population):,}')
display(population.head())

## 5. Controlled integration

The analytical spine is the OMI municipality-semester panel. Transaction and population measures are attached using municipality code + year. Missing values are retained and measured rather than imputed.

In [ ]:
market_panel = quotation_panel.merge(
    transactions[['year', 'municipality_code', 'ntn']],
    left_on=['reference_year', 'municipality_code'],
    right_on=['year', 'municipality_code'],
    how='left',
    validate='many_to_one',
)
market_panel = market_panel.drop(columns='year')
market_panel = market_panel.merge(
    population,
    left_on=['reference_year', 'municipality_code'],
    right_on=['year', 'municipality_code'],
    how='left',
    validate='many_to_one',
)
market_panel = market_panel.drop(columns='year')

assert not market_panel.duplicated(['reference_year', 'semester', 'municipality_code']).any()
market_panel['price_yoy_pct'] = (
    market_panel.sort_values(['municipality_code', 'reference_date'])
    .groupby('municipality_code')['median_price_m2']
    .pct_change()
    * 100
)
market_panel['population_yoy_pct'] = (
    market_panel.sort_values(['municipality_code', 'reference_year'])
    .groupby('municipality_code')['population']
    .pct_change()
    * 100
)
market_panel['ntn_yoy_pct'] = (
    market_panel.sort_values(['municipality_code', 'reference_year'])
    .groupby('municipality_code')['ntn']
    .pct_change()
    * 100
)

print(f'Integrated rows: {len(market_panel):,}')
display(market_panel.head())

## 6. Integration quality and coverage

Before interpreting relationships, measure how much of the analytical panel has all three dimensions available.

In [ ]:
coverage = pd.DataFrame({
    'metric': [
        'integrated rows',
        'unique municipality-semester keys',
        'rows with price',
        'rows with NTN',
        'rows with population',
        'rows with all three',
        'municipalities',
        'years',
        'semesters',
    ],
    'value': [
        len(market_panel),
        market_panel[['reference_year', 'semester', 'municipality_code']].drop_duplicates().shape[0],
        market_panel['median_price_m2'].notna().sum(),
        market_panel['ntn'].notna().sum(),
        market_panel['population'].notna().sum(),
        market_panel[['median_price_m2', 'ntn', 'population']].notna().all(axis=1).sum(),
        market_panel['municipality_code'].nunique(),
        market_panel['reference_year'].nunique(),
        market_panel['semester'].nunique(),
    ],
})
coverage['share_of_integrated_rows_pct'] = (
    coverage['value'].div(len(market_panel)).mul(100).round(2)
)
display(coverage)

missing_by_year = (
    market_panel.groupby('reference_year')
    .agg(
        price_missing_pct=('median_price_m2', lambda x: x.isna().mean() * 100),
        ntn_missing_pct=('ntn', lambda x: x.isna().mean() * 100),
        population_missing_pct=('population', lambda x: x.isna().mean() * 100),
    )
    .reset_index()
)
display(missing_by_year)

## 7. National price and transaction dynamics

The national price indicator is the median municipality quotation. NTN is aggregated as a sum because transaction volume has an additive interpretation. Population is aggregated as a sum across municipalities.

In [ ]:
national_market = (
    market_panel.groupby(['reference_year', 'semester', 'reference_date'], as_index=False)
    .agg(
        median_price_m2=('median_price_m2', 'median'),
        total_ntn=('ntn', 'sum'),
        total_population=('population', 'sum'),
        municipalities_price=('median_price_m2', 'count'),
    )
)
national_market['price_change_pct'] = national_market['median_price_m2'].pct_change() * 100
national_market['ntn_change_pct'] = national_market['total_ntn'].pct_change() * 100
display(national_market.tail(12))

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(national_market['reference_date'], national_market['median_price_m2'], marker='o')
ax.set_title('Italian Residential OMI Quotations — Median Municipality Price')
ax.set_xlabel('Reference period')
ax.set_ylabel('€ / m²')
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 8. Population growth versus price appreciation

The first cross-sectional test asks whether municipalities with stronger population growth also exhibit stronger price appreciation. Only observations with valid changes and both measures available are used.

In [ ]:
analysis = market_panel.dropna(
    subset=['price_yoy_pct', 'population_yoy_pct', 'ntn_yoy_pct']
).copy()
analysis = (
    analysis.replace([np.inf, -np.inf], np.nan)
    .dropna(subset=['price_yoy_pct', 'population_yoy_pct', 'ntn_yoy_pct'])
)

correlations = pd.DataFrame({
    'relationship': [
        'population growth vs price growth',
        'population growth vs NTN growth',
        'price growth vs NTN growth',
    ],
    'pearson_corr': [
        analysis['population_yoy_pct'].corr(analysis['price_yoy_pct']),
        analysis['population_yoy_pct'].corr(analysis['ntn_yoy_pct']),
        analysis['price_yoy_pct'].corr(analysis['ntn_yoy_pct']),
    ],
    'observations': [len(analysis)] * 3,
})
display(correlations)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(analysis['population_yoy_pct'], analysis['price_yoy_pct'], alpha=0.15)
ax.set_title('Population Growth vs Price Growth')
ax.set_xlabel('Population YoY change (%)')
ax.set_ylabel('OMI median price YoY change (%)')
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

## 9. Market segmentation

National correlations can hide substantial heterogeneity. Municipalities are therefore grouped into population-growth and price-growth quadrants.

In [ ]:
analysis['population_growth_group'] = np.where(
    analysis['population_yoy_pct'] >= 0,
    'Population growth',
    'Population decline',
)
analysis['price_growth_group'] = np.where(
    analysis['price_yoy_pct'] >= 0,
    'Price growth',
    'Price decline',
)
quadrants = (
    analysis.groupby(
        ['population_growth_group', 'price_growth_group'],
        as_index=False,
    )
    .agg(
        observations=('municipality_code', 'size'),
        municipalities=('municipality_code', 'nunique'),
        median_ntn_growth_pct=('ntn_yoy_pct', 'median'),
    )
    .sort_values('observations', ascending=False)
)
quadrants['share_pct'] = quadrants['observations'] / len(analysis) * 100
display(quadrants)

## 10. Largest markets and demographic context

A market-level view helps identify whether national patterns are driven by a small number of large municipalities.

In [ ]:
latest_year = market_panel['reference_year'].max()
latest = market_panel.loc[market_panel['reference_year'].eq(latest_year)].copy()
latest_summary = (
    latest.groupby('municipality_code', as_index=False)
    .agg(
        population=('population', 'first'),
        ntn=('ntn', 'first'),
        median_price_m2=('median_price_m2', 'median'),
        price_yoy_pct=('price_yoy_pct', 'median'),
        population_yoy_pct=('population_yoy_pct', 'median'),
    )
    .sort_values('population', ascending=False)
)
display(latest_summary.head(20))

## 11. Analytical conclusions and limitations

### What this panel enables
- compare price levels and transaction intensity at municipality level;
- examine demographic expansion or contraction alongside market dynamics;
- segment municipalities into price/population growth regimes;
- identify candidates for deeper regional or municipality-level case studies.

### Interpretation limits
- OMI quotations are valuation ranges, not observed transaction prices;
- the municipality price indicator is a descriptive median of OMI quotation midpoints;
- NTN measures normalized transaction volumes and does not represent transaction value;
- population is annual while OMI quotations are semiannual;
- changes in OMI coverage, municipality boundaries, property mix and data availability can affect comparisons;
- correlations do not establish causal effects.

### Next analytical step

The next notebook should move from descriptive integration to a **regional/municipality panel analysis**, controlling at minimum for market size and geography and separating price level, price growth and transaction intensity.